In [ ]:
from torchvision import transforms, datasets
from collections import Counter
from datetime import datetime
import torch
import math
import json
import os

# Datasets cropped sizes

In [5]:
class data_loader():
    def __init__(self, config):
        self.transform = transforms.Compose([transforms.ToTensor(), transforms.Resize(config["img_size"], antialias=True)])


    def __get_pytorch_dataset(self, subsets):
        dataset_list = []

        for subset in subsets:
            _, path = subset
            dataset_list.append(datasets.ImageFolder(path, self.transform))
        
        return torch.utils.data.ConcatDataset(dataset_list)
    

    def __get_dates(self, ds_path):
        dates = os.listdir(ds_path)
        formated_dates = [(datetime.strptime(date, '%Y-%m-%d').date(), f"{ds_path}/{date}") for date in dates]
        
        return sorted(formated_dates, key=lambda x: x[0])
    

    def load_dataset_by_split(self, ds_path="", t_size=1):
        assert os.path.exists(ds_path), "Invalid dataset"

        first_half, second_half = [], []

        subsets = [os.path.join(ds_path, subset) for subset in os.listdir(ds_path)]
        for subset in subsets:
            dates = self.__get_dates(subset)

            divider = math.ceil(len(dates) * t_size)
            first_half.extend(dates[:divider])
            second_half.extend(dates[divider:])
        
        if len(second_half) == 0:
            return self.__get_pytorch_dataset(first_half)
        else:
            return (self.__get_pytorch_dataset(first_half), self.__get_pytorch_dataset(second_half))

In [6]:
PKLOT_PATH = "/home/tteuz/Desktop/mestrado/datasets/segmented/PKLot"
CNRPARK_PATH = "/home/tteuz/Desktop/mestrado/datasets/segmented/CNRParkEXT"
PLDS_PATH = "/home/tteuz/Desktop/mestrado/datasets/segmented/PLds"

DL_CONFIG = { "img_size": [128, 128], "batch_size": 1000, "num_workers": 6 }

In [7]:
ds_loader = data_loader(DL_CONFIG)

full_pklot = ds_loader.load_dataset_by_split(PKLOT_PATH, 1)
full_cnrpark = ds_loader.load_dataset_by_split(CNRPARK_PATH, 1)
full_plds = ds_loader.load_dataset_by_split(PLDS_PATH, 1)

In [8]:
pklot_days, cnrpark_days, plds_days = set(), set(), set()

for subset in os.listdir(PKLOT_PATH):
    for day in os.listdir(f"{PKLOT_PATH}/{subset}"):
        pklot_days.add(day)

for subset in os.listdir(CNRPARK_PATH):
    for day in os.listdir(f"{CNRPARK_PATH}/{subset}"):
        cnrpark_days.add(day)

for subset in os.listdir(PLDS_PATH):
    for day in os.listdir(f"{PLDS_PATH}/{subset}"):
        plds_days.add(day)

In [9]:
pklot_empty, pklot_occupied, = 0, 0
cnrpark_empty, cnrpark_occupied = 0, 0
plds_empty, plds_occupied = 0, 0

for subset in full_pklot.datasets:
    count = Counter(subset.targets)
    pklot_empty += count[0]
    pklot_occupied += count[1]

for subset in full_cnrpark.datasets:
    count = Counter(subset.targets)
    cnrpark_empty += count[0]
    cnrpark_occupied += count[1]

for subset in full_plds.datasets:
    count = Counter(subset.targets)
    plds_empty += count[0]
    plds_occupied += count[1]

In [10]:
print(f"Tamanho PKLot: {len(full_pklot)} - Vazio: {pklot_empty} - Ocupado: {pklot_occupied} - Dias: {len(pklot_days)}")
print(f"Tamanho CNRPark: {len(full_cnrpark)} - Vazio: {cnrpark_empty} - Ocupado: {cnrpark_occupied} - Dias: {len(cnrpark_days)}")
print(f"Tamanho PLds: {len(full_plds)} - Vazio: {plds_empty} - Ocupado: {plds_occupied} - Dias: {len(plds_days)}")

Tamanho PKLot: 1199751 - Vazio: 656270 - Ocupado: 543481 - Dias: 100
Tamanho CNRPark: 148880 - Vazio: 66902 - Ocupado: 81978 - Dias: 23
Tamanho PLds: 104728 - Vazio: 60417 - Ocupado: 44311 - Dias: 71


# RAM data sizes

In [11]:
PKLOT_RAW_PATH = "/home/tteuz/Desktop/mestrado/datasets/organized/PKLot"
CNRPARK_RAW_PATH = "/home/tteuz/Desktop/mestrado/datasets/organized/CNRParkEXT"
PLDS_RAW_PATH = "/home/tteuz/Desktop/mestrado/datasets/organized/PLds"

In [12]:
def get_crop_artifacts(json_path):
    with open(json_path, "r") as file:
        data = json.load(file)

    images_ids = set()
    for annotation in data["annotations"]:
        images_ids.add(annotation["image_id"])
    
    annotations = {id: [] for id in images_ids}
    for annotation in data["annotations"]:
        annotations[annotation["image_id"]].append(annotation)

    return (data["images"], annotations, data["categories"])

In [13]:
def get_images_qty(json_path):
    jsons = [file for file in os.listdir(json_path) if ".json" in file]

    images_qty = 0
    for file in jsons:
        artifacts = get_crop_artifacts(f"{json_path}/{file}")

        images_qty += len(artifacts[0])

    return images_qty

In [14]:
print(f"PKLot total images: {get_images_qty(PKLOT_RAW_PATH)}")
print(f"CNRPark-Ext total images: {get_images_qty(CNRPARK_RAW_PATH)}")
print(f"PLds total images: {get_images_qty(PLDS_RAW_PATH)}")

PKLot total images: 12400
CNRPark-Ext total images: 4073
PLds total images: 8188
